In [ ]:
!pip -q install transformers accelerate peft bitsandbytes trl datasets evaluate sentencepiece

In [ ]:
import gc
import random
import torch

from peft import LoraConfig, get_peft_model, PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset, concatenate_datasets
from trl import DPOTrainer, DPOConfig, KTOTrainer, KTOConfig


SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

In [ ]:
import os
import warnings

os.environ["WANDB_DISABLED"] = "true"
warnings.filterwarnings('ignore')

### DPO

In [ ]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
base = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map="auto")

lora_cfg = LoraConfig(
    task_type="CAUSAL_LM",
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","up_proj","down_proj","gate_proj"]
)
policy = get_peft_model(base, lora_cfg)
policy.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [ ]:
def chat(model, prompt, max_new_tokens=128):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, do_sample=True, top_p=0.9, temperature=0.7, max_new_tokens=max_new_tokens)
    return tokenizer.decode(out[0], skip_special_tokens=True)

In [ ]:
print(chat(policy, "Коротко объясни разницу между DPO и PPO."))

Коротко объясни разницу между DPO и PPO. | 360 Курса
14-25-2022, 22:08
Для понимания, как они различаются, стоит посвятиться 10 минут. Дополнительные материалы можно найти в общей групповой программе обучения.
Как известно, PPO - это метод, который использует алгоритмы машинного обучения для прогнозирования. В отличие от DPO - это упрощенный подход к оценке качества.
Вот что значит: ПОП - это метод, который вычисляет вероят


In [ ]:
prompts = [
  "Объясни, что такое over-refusal, в 2–3 предложениях.",
  "Explain DPO vs PPO in one paragraph.",
  "Сделай краткий список шагов для обучения PPO.",
  "Make a short list of steps for training PPO."
]
baseline_outputs = [chat(policy, p, 128) for p in prompts]

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def build_dpo_row(ex):
    msgs_base = [
        {"role":"system","content":"You are a helpful, safe assistant."},
        {"role":"user","content": ex["prompt"]}
    ]
    prompt_text = tokenizer.apply_chat_template(msgs_base, tokenize=False, add_generation_prompt=True)
    msgs_ch = msgs_base + [{"role":"assistant","content": ex["chosen"][1]['content']}]
    msgs_rj = msgs_base + [{"role":"assistant","content": ex["rejected"][1]['content']}]
    chosen_text   = tokenizer.apply_chat_template(msgs_ch, tokenize=False, add_generation_prompt=False)
    rejected_text = tokenizer.apply_chat_template(msgs_rj, tokenize=False, add_generation_prompt=False)
    return {"prompt": prompt_text, "chosen": chosen_text, "rejected": rejected_text}

ds = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs[:2000]")
dpo_ds = ds.map(build_dpo_row, remove_columns=ds.column_names)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
dpo_ds

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 2000
})

In [ ]:
ref = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map="auto")
ref.eval()

cfg = DPOConfig(
    beta=0.05,
    learning_rate=5e-6, per_device_train_batch_size=1, gradient_accumulation_steps=8,
    max_steps=100, warmup_ratio=0.1, bf16=True, logging_steps=10, optim="paged_adamw_8bit"
)
trainer = DPOTrainer(model=policy, ref_model=ref, args=cfg, train_dataset=dpo_ds, processing_class=tokenizer)
trainer.train()
policy.save_pretrained("qwen_dpo_lora")

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Extracting prompt in train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss
10,0.692600
20,0.685100
30,0.691200
40,0.694100
50,0.690800
60,0.692000
70,0.679500
80,0.690100
90,0.692800
100,0.674100


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [ ]:
base = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map="auto")
policy = PeftModel.from_pretrained(base, "qwen_dpo_lora")

dpo_outputs = [chat(policy, p, 128) for p in prompts]
for i,p in enumerate(prompts[:5]):
    print("Q:", p, "\nBASE:", baseline_outputs[i][:200], "\nDPO:", dpo_outputs[i][:200], "\n---")

Q: Объясни, что такое over-refusal, в 2–3 предложениях. 
BASE: Объясни, что такое over-refusal, в 2–3 предложениях. Основные характеристики и преимущества этого состояния

over-refusal (англ. Over-acceptance) – это состояние, когда люди не принимают уважения или  
DPO: Объясни, что такое over-refusal, в 2–3 предложениях. Основные особенности, с чем отличается от описания и несовершенного речи.

Over-refusal - это русское слово, которое означает отказаться от чего-ли 
---
Q: Explain DPO vs PPO in one paragraph. 
BASE: Explain DPO vs PPO in one paragraph. DPO stands for Deep Q-Learning Policy Optimization, which is a type of reinforcement learning algorithm. It is used to optimize the parameters of a policy network  
DPO: Explain DPO vs PPO in one paragraph. The DPO (Double-Period) and PPO (Properly-Defined-Optimally) approaches are both variants of the Policy Gradient Model, which is a popular approach for training ne 
---
Q: Сделай краткий список шагов для обучения PPO. 
BASE: Сделай к

In [ ]:
import gc

In [ ]:
del ref, trainer, policy, base

gc.collect()
torch.cuda.empty_cache()

[KTOTrainer](https://huggingface.co/docs/trl/main/kto_trainer)